# 16 - Gold 5: reconciliation (TC08)
Each Gold number is compared with an **independent query on Silver** at the same grain. Results are saved to
`gold_reconciliation` with expected, actual and PASS / FAIL.

In [0]:
from pyspark.sql import functions as F

cat = "capstone6_telecom.telecom"

def n(sql):
    return spark.sql(sql).collect()[0][0]

checks = []
def check(test, metric, actual, expected):
    a, e = int(actual), int(expected)
    checks.append((test, metric, a, e, "PASS" if a == e else "FAIL"))

## Record counts and grain

In [0]:
silver_events = n(f"SELECT COUNT(*) FROM {cat}.silver_network_events")
silver_failed = n(f"SELECT COUNT(*) FROM {cat}.silver_network_current WHERE current_status = 'FAILED'")

check("GRAIN", "fact_network_activity rows = silver events", n(f"SELECT COUNT(*) FROM {cat}.fact_network_activity"), silver_events)
check("GRAIN", "fact_network_activity duplicate measurement_id", n(f"SELECT COUNT(*) - COUNT(DISTINCT measurement_id) FROM {cat}.fact_network_activity"), 0)
check("GRAIN", "fact_status_transitions rows = silver history", n(f"SELECT COUNT(*) FROM {cat}.fact_status_transitions"), n(f"SELECT COUNT(*) FROM {cat}.silver_status_history"))
check("GRAIN", "fact_status_transitions duplicate keys", n(f"SELECT COUNT(*) - COUNT(DISTINCT measurement_id, source_sequence) FROM {cat}.fact_status_transitions"), 0)
check("GRAIN", "fact_outage_interval rows = silver intervals", n(f"SELECT COUNT(*) FROM {cat}.fact_outage_interval"), n(f"SELECT COUNT(*) FROM {cat}.silver_outage_intervals"))
check("GRAIN", "mart duplicate (date, region)", n(f"SELECT COUNT(*) - COUNT(DISTINCT event_date, region_id) FROM {cat}.mart_network_daily"), 0)
check("GRAIN", "dim_region duplicate keys", n(f"SELECT COUNT(*) - COUNT(DISTINCT region_key) FROM {cat}.dim_region"), 0)
check("GRAIN", "dim_tower duplicate keys", n(f"SELECT COUNT(*) - COUNT(DISTINCT tower_key) FROM {cat}.dim_tower"), 0)

## Joins: every fact row finds its dimension rows

In [0]:
check("JOIN", "fact rows with missing region/tower/date key", n(f"""
    SELECT COUNT(*) FROM {cat}.fact_network_activity f
    LEFT JOIN {cat}.dim_date d ON f.date_key = d.date_key
    WHERE f.region_key IS NULL OR f.tower_key IS NULL OR d.date_key IS NULL"""), 0)
check("JOIN", "transition rows with missing keys", n(f"""
    SELECT COUNT(*) FROM {cat}.fact_status_transitions f
    LEFT JOIN {cat}.dim_date d ON f.date_key = d.date_key
    WHERE f.region_key IS NULL OR f.tower_key IS NULL OR d.date_key IS NULL"""), 0)

## KPI totals: Gold mart vs Silver

In [0]:
check("TC08", "mart total readings = silver events", n(f"SELECT SUM(readings) FROM {cat}.mart_network_daily"), silver_events)
check("TC08", "mart outage readings = silver current FAILED", n(f"SELECT SUM(outage_readings) FROM {cat}.mart_network_daily"), silver_failed)

## KPI by region-day: independent query on Silver (expect 0 mismatching region-days)

In [0]:
mismatch = n(f"""
WITH s AS (
  SELECT CAST(event_ts AS DATE) AS event_date, region_id, current_status, packet_loss_pct, tower_id, event_ts, measurement_id,
         LEAD(event_ts) OVER (PARTITION BY tower_id ORDER BY event_ts, measurement_id) AS nxt
  FROM {cat}.silver_network_enriched
), indep AS (
  SELECT event_date, region_id, COUNT(*) AS readings,
         SUM(CASE WHEN current_status = 'FAILED' THEN 1 ELSE 0 END) AS outage_readings,
         SUM(packet_loss_pct) AS packet_loss_sum,
         SUM(CASE WHEN current_status = 'FAILED' AND nxt IS NOT NULL
                  THEN (unix_timestamp(nxt) - unix_timestamp(event_ts)) / 60.0 ELSE 0 END) AS outage_minutes
  FROM s GROUP BY event_date, region_id
)
SELECT COUNT(*) FROM indep i
FULL OUTER JOIN {cat}.mart_network_daily m ON i.event_date = m.event_date AND i.region_id = m.region_id
WHERE i.event_date IS NULL OR m.event_date IS NULL
   OR i.readings <> m.readings OR i.outage_readings <> m.outage_readings
   OR ABS(i.packet_loss_sum - m.packet_loss_sum) > 0.001
   OR ABS(i.outage_minutes - m.outage_minutes) > 0.01""")
check("TC08", "region-days where Gold differs from independent Silver query", mismatch, 0)

## Outage minutes: every closed Silver interval equals the sum of its fact readings (expect 0 mismatches)

In [0]:
bad_intervals = n(f"""
SELECT COUNT(*) FROM (
  SELECT o.outage_id, o.outage_minutes AS interval_minutes, SUM(f.outage_minutes) AS fact_minutes
  FROM {cat}.silver_outage_intervals o
  JOIN {cat}.fact_network_activity f
    ON f.tower_id = o.tower_id AND f.is_outage_reading
   AND f.event_ts BETWEEN o.start_ts AND o.last_failed_ts
  WHERE NOT o.is_open
  GROUP BY o.outage_id, o.outage_minutes
) WHERE ABS(interval_minutes - fact_minutes) > 0.01""")
check("TC08", "closed intervals whose minutes differ from fact", bad_intervals, 0)

check("TC08", "interval failed readings = fact outage readings",
      n(f"SELECT SUM(failed_readings) FROM {cat}.fact_outage_interval"),
      n(f"SELECT COUNT(*) FROM {cat}.fact_network_activity WHERE is_outage_reading"))

## Save the evidence (every row should say PASS)

In [0]:
(spark.createDataFrame(checks, "test_id string, metric string, actual long, expected long, result string")
      .withColumn("checked_at", F.current_timestamp())
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{cat}.gold_reconciliation"))
display(spark.table(f"{cat}.gold_reconciliation"))